# NoteHut OCR Worker Setup

Sets up a GPU-accelerated OCR worker for NoteHut on Google Colab.
This notebook installs dependencies, starts Ollama (embeddings + LLM),
launches the FastAPI OCR worker, and creates a public tunnel via Cloudflare.

**Prerequisites:**
- A Supabase project with the NoteHut schema applied
- Your Supabase URL and service role key
- The NoteHut repo cloned to `/content/Notehut` (or upload worker/ files manually)


In [ ]:
# Clone the NoteHut repo (for the worker code)
!git clone -q https://github.com/magdiemad5s/Notehut.git /content/Notehut 2>/dev/null || echo "Repo already cloned"

# Install cloudflared (for tunnel), tesseract (fallback OCR), and zstd (Ollama dependency)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!apt-get update -qq && apt-get install -y -qq tesseract-ocr zstd
print("System deps installed, repo cloned")

In [ ]:
!pip install -q fastapi uvicorn[standard] supabase pymupdf httpx transformers torch Pillow einops addict easydict psutil pyngrok
print("Python deps installed")

## Configuration — Fill in your secrets

In [ ]:
import os

# === FILL IN YOUR SECRETS ===
os.environ["SUPABASE_URL"] = ""           # Your Supabase project URL
os.environ["SUPABASE_SERVICE_KEY"] = ""   # Supabase service role key
os.environ["WORKER_API_KEY"] = ""          # Choose a secret key for worker auth (optional)

# === Tunnel choice ===
TUNNEL = "cloudflare"  # "cloudflare" or "ngrok"
NGROK_TOKEN = ""        # Only if using ngrok

print("Configuration set")

## Start Ollama + Pull Models

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in background (fully detached with nohup, logs to file)
# nohup + & detaches it from the cell so it doesn't block
!nohup ollama serve > /content/ollama.log 2>&1 &

import time
time.sleep(5)  # wait for Ollama to bind port 11434

# Verify Ollama is responding before pulling
!curl -s http://localhost:11434/api/version

# Pull models (may take a few minutes)
!ollama pull qwen3-embedding:0.6b
!ollama pull qwen3.5:9b   # Q4_K_M quant (6.6 GB), Ollama default for 9B — industry sweet spot

print("Ollama started and models pulled")

## Start OCR Worker

In [ ]:
import subprocess, sys, os, glob

# Find the worker directory (supports both repo clone and manual upload)
candidates = ["/content/Notehut/worker", "/content/worker", "worker", "."]
worker_dir = next((d for d in candidates if os.path.isfile(os.path.join(d, "ocr_worker.py"))), ".")
print(f"Worker dir: {os.path.abspath(worker_dir)}")

# Start the worker in background, redirect output to a log file for debugging
log_file = open("/content/worker.log", "w")
worker_proc = subprocess.Popen(
    [sys.executable, "ocr_worker.py"],
    cwd=worker_dir,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ},
)

import time
time.sleep(5)
if worker_proc.poll() is not None:
    print("ERROR: Worker crashed! Log output:")
    print(open("/content/worker.log").read())
else:
    print("OCR worker started on port 8000")
    print("Logs at /content/worker.log")

## Start Tunnel

In [ ]:
import subprocess, time, re, sys

if TUNNEL == "cloudflare":
    # Start cloudflared tunnel
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )

    # Wait for the tunnel URL to appear in output
    tunnel_url = None
    for line in iter(tunnel_proc.stdout.readline, ''):
        print(line, end='')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
        if tunnel_proc.poll() is not None:
            break

    if tunnel_url:
        print(f"\n{'='*60}")
        print(f"TUNNEL URL: {tunnel_url}")
        print(f"{'='*60}")
        print(f"\nPaste this in admin panel:")
        print(f"  Worker URL:        {tunnel_url}")
        print(f"  Embeddings Base:   {tunnel_url}/ollama/v1")
        print(f"  LLM Base URL:      {tunnel_url}/ollama/v1")
        print(f"  LLM Model:         qwen3.5:9b")
        print(f"  Embeddings Model:  qwen3-embedding:0.6b")
        print(f"  Worker log:        /content/worker.log")
    else:
        print("Failed to get tunnel URL")

elif TUNNEL == "ngrok":
    from pyngrok import ngrok
    if NGROK_TOKEN:
        ngrok.set_auth_token(NGROK_TOKEN)
    tunnel_url = ngrok.connect(8000).public_url
    print(f"\n{'='*60}")
    print(f"TUNNEL URL: {tunnel_url}")
    print(f"{'='*60}")
    print(f"\nPaste this in admin panel:")
    print(f"  Worker URL:        {tunnel_url}")
    print(f"  Embeddings Base:   {tunnel_url}/ollama/v1")
    print(f"  LLM Base URL:      {tunnel_url}/ollama/v1")
    print(f"  LLM Model:         qwen3.5:9b")
    print(f"  Embeddings Model:  qwen3-embedding:0.6b")
    print(f"  Worker log:        /content/worker.log")

## Keep Alive — Run this to prevent Colab timeout

In [ ]:
import time, subprocess
from IPython.display import clear_output

while True:
    time.sleep(60)
    clear_output(wait=True)
    print(f"Worker running... {time.strftime('%H:%M:%S')}")
    # Check if worker process is alive
    if worker_proc.poll() is not None:
        print("WARNING: Worker process died!")
    # Check Ollama via HTTP (it's a nohup process, not a Popen)
    try:
        subprocess.run(["curl", "-s", "http://localhost:11434/api/version"], timeout=5, capture_output=True, check=True)
    except Exception:
        print("WARNING: Ollama not responding!")
    # Check tunnel process (cloudflare only)
    if TUNNEL == "cloudflare" and 'tunnel_proc' in globals() and tunnel_proc.poll() is not None:
        print("WARNING: Tunnel process died!")